# Model 4 — MLP-Mixer (PillSight)
**ADSP 31018 | University of Chicago**

Architecture: MLP-Mixer (Tolstikhin et al., 2021)  
Dataset: ePillID — top 10 pill classes, 2312 images

### Steps
1. Download dataset
2. Clone repo
3. Train
4. Evaluate
5. Visualise patch importance

> **Runtime → Change runtime type → T4 GPU** before running.

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 1. Install dependencies

In [ ]:
!pip install torch torchvision scikit-learn pandas Pillow matplotlib -q

## 2. Download ePillID dataset
Download from the official release and unzip.

In [ ]:
!wget -q --show-progress https://github.com/usuyama/ePillID-benchmark/releases/download/ePillID_data_v1.0/ePillID_data.zip
!unzip -q ePillID_data.zip
# zip extracts to ePillID_data/ — show contents
!ls ePillID_data/

## 3. Clone repo and set up paths

In [ ]:
!git clone https://github.com/Devanshu1503/ML2_Class_Project.git

import os, shutil

# zip extracts to ePillID_data/ — move its contents into ML2_Class_Project/data/
os.makedirs('ML2_Class_Project/data', exist_ok=True)
for item in os.listdir('ePillID_data'):
    shutil.move(f'ePillID_data/{item}', f'ML2_Class_Project/data/{item}')

print('data folder contents:')
!ls ML2_Class_Project/data/
print()
print('repo structure:')
!ls ML2_Class_Project/

## 4. Generate filtered dataset (run model1's data_exploration.py — shared step)

In [ ]:
%cd ML2_Class_Project
!python model1_cnn/data_exploration.py

## 5. Config

In [ ]:
import sys
sys.path.insert(0, 'model4_mlpmixer')
import config as cfg

print('DATA_ROOT    :', cfg.DATA_ROOT)
print('NUM_CLASSES  :', cfg.NUM_CLASSES)
print('PATCH_SIZE   :', cfg.PATCH_SIZE)
print('HIDDEN_DIM   :', cfg.HIDDEN_DIM)
print('MIXER_LAYERS :', cfg.NUM_MIXER_LAYERS)
print('BATCH_SIZE   :', cfg.BATCH_SIZE)
print('EPOCHS       :', cfg.NUM_EPOCHS)
print('LR           :', cfg.LEARNING_RATE)

## 6. Model architecture

In [ ]:
from model4_mlpmixer.model import PillMixer, count_parameters

model = PillMixer(num_classes=cfg.NUM_CLASSES)
dummy = torch.zeros(1, 3, cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)
out   = model(dummy)

print('Output shape :', out.shape)
print('Num patches  :', (cfg.IMAGE_SIZE // cfg.PATCH_SIZE) ** 2)
print('Parameters   :', count_parameters(model))

## 7. Train

In [ ]:
!python model4_mlpmixer/train.py

## 8. Evaluate on test set

In [ ]:
!python model4_mlpmixer/evaluate.py

## 9. Patch importance visualisation

In [ ]:
!python model4_mlpmixer/patch_importance.py

## 10. Show results

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

# Print metrics
metrics_path = Path('model4_mlpmixer/outputs/metrics.json')
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    print('=== Model 4 — MLP-Mixer Results ===')
    print(f"Top-1 Accuracy : {m['top1']*100:.2f}%")
    print(f"Top-3 Accuracy : {m['top3']*100:.2f}%")
    print(f"Macro F1       : {m['macro_f1']:.4f}")

# Learning curves
lc = Path('model4_mlpmixer/outputs/learning_curves.png')
if lc.exists():
    plt.figure(figsize=(12,4))
    plt.imshow(mpimg.imread(str(lc)))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Confusion matrix
cm = Path('model4_mlpmixer/outputs/confusion_matrix.png')
if cm.exists():
    plt.figure(figsize=(8,8))
    plt.imshow(mpimg.imread(str(cm)))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 11. Patch importance samples

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

viz_dir = Path('model4_mlpmixer/outputs/patch_importance')
images  = sorted(viz_dir.glob('*.png'))

for img_path in images:
    plt.figure(figsize=(12, 4))
    plt.imshow(mpimg.imread(str(img_path)))
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 12. Save results back to Drive (optional)

In [ ]:
# Uncomment to save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r model4_mlpmixer/outputs /content/drive/MyDrive/ML2_Project_Model4_Outputs
# !cp model4_mlpmixer/checkpoints/best_model.pth /content/drive/MyDrive/model4_best_model.pth
print('Uncomment the lines above to save outputs to Google Drive')